# 03 — Silver and Gold
Validates Bronze records, quarantines invalid data, and publishes dimensions and aggregates.

In [ ]:
import os
import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, IntegerType, StringType, StructField, StructType, TimestampType

catalog = os.getenv('AIDP_CATALOG', 'aidp_poc')
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', catalog):
    raise ValueError('AIDP_CATALOG must be a simple Spark identifier')
try:
    late_arrival_hours = int(os.getenv('LATE_ARRIVAL_HOURS', '48'))
except ValueError as error:
    raise ValueError('LATE_ARRIVAL_HOURS must be an integer') from error
if late_arrival_hours < 0:
    raise ValueError('LATE_ARRIVAL_HOURS must be zero or greater')

schema = StructType([
    StructField('schema_version', StringType()), StructField('event_id', StringType()), StructField('event_type', StringType()), StructField('event_ts_utc', TimestampType()),
    StructField('meter_id', StringType()), StructField('device_id', StringType()), StructField('service_point_id', StringType()), StructField('service_point_type', StringType()),
    StructField('interval_start_utc', TimestampType()), StructField('interval_end_utc', TimestampType()), StructField('interval_minutes', IntegerType()), StructField('consumption_kwh', DoubleType()),
    StructField('voltage_v', DoubleType()), StructField('current_a', DoubleType()), StructField('power_factor', DoubleType()), StructField('temperature_c', DoubleType()),
    StructField('quality_code', StringType()), StructField('tariff_code', StringType()), StructField('measurement_events', ArrayType(StringType())), StructField('device_events', ArrayType(StringType()))
])

bronze = f'{catalog}.bronze.meter_reading'
quarantine = f'{catalog}.bronze.meter_reading_quarantine'
silver_table = f'{catalog}.silver.interval_reading'
gold_interval = f'{catalog}.gold.meter_interval_usage'
gold_daily = f'{catalog}.gold.service_point_daily_usage'
device_table = f'{catalog}.silver.device'
service_point_table = f'{catalog}.silver.service_point'
device_event_table = f'{catalog}.silver.device_event'


In [ ]:
raw = spark.table(bronze).select('stream_partition', 'stream_offset', 'stream_key', 'raw_value')
checked = raw.withColumn('payload', F.from_json('raw_value', schema)).withColumn('ingested_at', F.current_timestamp())
payload = F.col('payload')
checked = checked.withColumn('validation_reason', F.concat_ws('; ',
    F.when(payload.isNull(), F.lit('INVALID_JSON_OR_SCHEMA')),
    F.when(payload.event_id.isNull() | payload.meter_id.isNull() | payload.device_id.isNull() | payload.service_point_id.isNull() | payload.interval_start_utc.isNull() | payload.interval_end_utc.isNull() | payload.interval_minutes.isNull() | payload.consumption_kwh.isNull() | payload.quality_code.isNull() | payload.tariff_code.isNull(), F.lit('MISSING_REQUIRED_FIELD')),
    F.when(payload.event_type != 'INTERVAL_READING', F.lit('UNSUPPORTED_EVENT_TYPE')),
    F.when(payload.interval_minutes != 15, F.lit('INVALID_INTERVAL_MINUTES')),
    F.when((payload.consumption_kwh < 0) | (payload.consumption_kwh > 1000), F.lit('INVALID_CONSUMPTION')),
    F.when((payload.power_factor < -1) | (payload.power_factor > 1), F.lit('INVALID_POWER_FACTOR')),
    F.when(payload.interval_end_utc != payload.interval_start_utc + F.expr('INTERVAL 15 MINUTES'), F.lit('INVALID_INTERVAL_BOUNDARY')),
    F.when(~payload.quality_code.isin('ACTUAL', 'ESTIMATED', 'SUBSTITUTED', 'SUSPECT'), F.lit('INVALID_QUALITY_CODE'))
))
checked = checked.withColumn('validation_reason', F.when((F.col('validation_reason') == '') & (F.col('payload.interval_start_utc') < F.current_timestamp() - F.expr(f'INTERVAL {late_arrival_hours} HOURS')), F.lit('LATE_BEYOND_ALLOWED_WINDOW')).otherwise(F.col('validation_reason')))

bad = checked.where("validation_reason <> ''").select('stream_partition', 'stream_offset', 'stream_key', 'raw_value', 'validation_reason', F.current_timestamp().alias('quarantined_at'))
bad.createOrReplaceTempView('bad_batch')
spark.sql(f'MERGE INTO {quarantine} t USING bad_batch s ON t.stream_partition=s.stream_partition AND t.stream_offset=s.stream_offset WHEN NOT MATCHED THEN INSERT *')

silver = checked.where("validation_reason = ''").select('payload.*', 'stream_partition', 'stream_offset', 'stream_key', 'ingested_at', F.to_date('payload.interval_start_utc').alias('reading_date'), F.when(F.col('payload.quality_code') == 'ACTUAL', F.lit(True)).otherwise(F.lit(False)).alias('is_actual'))
silver.createOrReplaceTempView('silver_batch')
spark.sql(f'MERGE INTO {silver_table} t USING silver_batch s ON t.event_id=s.event_id WHEN NOT MATCHED THEN INSERT *')


In [ ]:
spark.sql(f'''MERGE INTO {device_table} t USING (SELECT device_id, meter_id, service_point_id, service_point_type, min(interval_start_utc) first_seen_at, max(interval_start_utc) last_seen_at, max_by(event_id, interval_start_utc) last_event_id, current_timestamp() updated_at FROM {silver_table} GROUP BY device_id, meter_id, service_point_id, service_point_type) s ON t.device_id=s.device_id WHEN MATCHED THEN UPDATE SET meter_id=s.meter_id, service_point_id=s.service_point_id, service_point_type=s.service_point_type, last_seen_at=s.last_seen_at, last_event_id=s.last_event_id, updated_at=s.updated_at WHEN NOT MATCHED THEN INSERT *''')
spark.sql(f'''MERGE INTO {service_point_table} t USING (SELECT service_point_id, service_point_type, max_by(meter_id, interval_start_utc) latest_meter_id, min(interval_start_utc) first_seen_at, max(interval_start_utc) last_seen_at, current_timestamp() updated_at FROM {silver_table} GROUP BY service_point_id, service_point_type) s ON t.service_point_id=s.service_point_id WHEN MATCHED THEN UPDATE SET service_point_type=s.service_point_type, latest_meter_id=s.latest_meter_id, last_seen_at=s.last_seen_at, updated_at=s.updated_at WHEN NOT MATCHED THEN INSERT *''')
spark.sql(f'''MERGE INTO {device_event_table} t USING (SELECT event_id, meter_id, device_id, service_point_id, interval_start_utc, explode(device_events) device_event_type, ingested_at, reading_date FROM {silver_table}) s ON t.event_id=s.event_id AND t.device_event_type=s.device_event_type WHEN NOT MATCHED THEN INSERT *''')
spark.sql(f'''CREATE OR REPLACE TEMP VIEW gold_interval AS SELECT reading_date, interval_start_utc, meter_id, service_point_id, tariff_code, sum(consumption_kwh) consumption_kwh, avg(voltage_v) avg_voltage_v, avg(power_factor) avg_power_factor, count(*) reading_count, sum(CASE WHEN is_actual THEN 1 ELSE 0 END) actual_reading_count, sum(CASE WHEN quality_code='SUSPECT' THEN 1 ELSE 0 END) suspect_reading_count, current_timestamp() refreshed_at FROM {silver_table} GROUP BY reading_date, interval_start_utc, meter_id, service_point_id, tariff_code''')
spark.sql(f'INSERT OVERWRITE {gold_interval} SELECT * FROM gold_interval')
spark.sql(f'''INSERT OVERWRITE {gold_daily} SELECT reading_date, service_point_id, sum(consumption_kwh), sum(CASE WHEN tariff_code='PEAK' THEN consumption_kwh ELSE 0 END), sum(CASE WHEN tariff_code='OFF_PEAK' THEN consumption_kwh ELSE 0 END), count(*), sum(actual_reading_count), max(consumption_kwh), current_timestamp() FROM {gold_interval} GROUP BY reading_date, service_point_id''')
print('Silver, dimensions, and Gold tables refreshed.')